In [ ]:
# K-Means Segmentation
# 
# This notebook implements K-means color clustering segmentation on all 80 preprocessed images.
# 
# K-Means overview:
# - Unsupervised clustering of pixel colors into K groups
# - Each pixel is assigned to the cluster with the nearest centroid
# - Foreground/background determined by border heuristic
# 
# We test different values of K and color spaces.

In [ ]:
# Part 1: Setup

import numpy as np
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import json
import time

# Import preprocessing module
from preprocessing import load_image_pair

# Configure matplotlib
plt.rcParams['figure.figsize'] = (15, 5)

print("environment setup complete")

# Define paths
PROJECT_DATA = Path('../project_data')
PREPROCESSED_DIR = PROJECT_DATA / 'preprocessed'
RESULTS_DIR = Path('../results/kmeans')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Load manifest
df_manifest = pd.read_csv(PROJECT_DATA / 'selected_images.csv')

print(f"total images: {len(df_manifest)}")
print(f"results directory: {RESULTS_DIR}")

In [ ]:
# Part 2: K-Means Implementation

def segment_kmeans(image, k=2, color_space='lab', use_spatial=False, spatial_weight=0.5):
    """
    Segment image using K-means color clustering.
    
    Args:
        image: input RGB image (H x W x 3), uint8
        k: number of clusters
        color_space: 'rgb', 'hsv', or 'lab'
        use_spatial: whether to append (x,y) coordinates as features
        spatial_weight: weight for spatial features relative to color
    
    Returns:
        mask: binary segmentation mask (0 or 255)
        labels_img: cluster labels reshaped to image dimensions
    """
    h, w = image.shape[:2]
    
    # Convert color space
    if color_space == 'lab':
        converted = cv2.cvtColor(image, cv2.COLOR_RGB2Lab)
    elif color_space == 'hsv':
        converted = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    else:  # rgb
        converted = image.copy()
    
    # Reshape to 2D array of pixels (N x 3)
    pixel_vals = converted.reshape((-1, 3)).astype(np.float32)
    
    # Optionally add spatial coordinates
    if use_spatial:
        yy, xx = np.mgrid[0:h, 0:w]
        xx_norm = (xx.reshape(-1, 1) / w * 255 * spatial_weight).astype(np.float32)
        yy_norm = (yy.reshape(-1, 1) / h * 255 * spatial_weight).astype(np.float32)
        pixel_vals = np.hstack([pixel_vals, xx_norm, yy_norm])
    
    # K-means criteria: stop after 100 iterations or epsilon 0.85
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.85)
    
    # Run K-means
    compactness, labels, centers = cv2.kmeans(
        pixel_vals, k, None, criteria,
        attempts=10,
        flags=cv2.KMEANS_PP_CENTERS
    )
    
    # Reshape labels back to image shape
    labels_img = labels.reshape(h, w)
    
    # Determine foreground vs background using border heuristic:
    # The most common cluster on the image border is likely background
    border_labels = np.concatenate([
        labels_img[0, :],
        labels_img[-1, :],
        labels_img[:, 0],
        labels_img[:, -1]
    ])
    bg_label = np.bincount(border_labels.flatten().astype(int)).argmax()
    
    # Everything not background = foreground
    mask = np.where(labels_img != bg_label, 255, 0).astype(np.uint8)
    
    return mask, labels_img


# Test on single image
test_idx = 0
test_row = df_manifest.iloc[test_idx]

img_path = PREPROCESSED_DIR / 'images' / test_row['filename']
mask_path = PREPROCESSED_DIR / 'masks' / test_row['mask_filename']

test_img, test_gt = load_image_pair(img_path, mask_path)

# Test different color spaces
mask_rgb, _ = segment_kmeans(test_img, k=2, color_space='rgb')
mask_lab, _ = segment_kmeans(test_img, k=2, color_space='lab')
mask_hsv, _ = segment_kmeans(test_img, k=2, color_space='hsv')

# Visualize
fig, axes = plt.subplots(1, 5, figsize=(20, 4))

axes[0].imshow(test_img)
axes[0].set_title('Input image')
axes[0].axis('off')

axes[1].imshow(mask_rgb, cmap='gray')
axes[1].set_title('K-means (RGB)')
axes[1].axis('off')

axes[2].imshow(mask_lab, cmap='gray')
axes[2].set_title('K-means (Lab)')
axes[2].axis('off')

axes[3].imshow(mask_hsv, cmap='gray')
axes[3].set_title('K-means (HSV)')
axes[3].axis('off')

axes[4].imshow(test_gt, cmap='gray')
axes[4].set_title('Ground truth')
axes[4].axis('off')

plt.suptitle(f'Color Space Comparison - {test_row["subset"]}', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'colorspace_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"test complete on: {test_row['filename']}")

In [ ]:
# Part 3: Parameter Tuning - number of clusters K

k_values = [2, 3, 4, 5]

fig, axes = plt.subplots(1, len(k_values) + 1, figsize=(20, 4))

axes[0].imshow(test_img)
axes[0].set_title('input')
axes[0].axis('off')

for i, k in enumerate(k_values):
    mask, _ = segment_kmeans(test_img, k=k, color_space='lab')
    
    axes[i+1].imshow(mask, cmap='gray')
    axes[i+1].set_title(f'K = {k}')
    axes[i+1].axis('off')

plt.suptitle('K-Means: Cluster Count Comparison (Lab color space)', fontsize=14)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'k_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Part 3b: Quick quantitative check across a few images to pick best config

def evaluate_quick(pred_mask, gt_mask):
    """Quick IoU calculation."""
    pred = (pred_mask > 127).astype(np.uint8)
    gt = (gt_mask > 127).astype(np.uint8)
    intersection = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return intersection / union if union > 0 else 0.0

# Test on first 10 images for speed
n_test = 10
k_values = [2, 3, 4, 5]
color_spaces = ['rgb', 'lab', 'hsv']

tuning_results = []

for idx in tqdm(range(min(n_test, len(df_manifest))), desc="tuning"):
    row = df_manifest.iloc[idx]
    img_path = PREPROCESSED_DIR / 'images' / row['filename']
    mask_path = PREPROCESSED_DIR / 'masks' / row['mask_filename']
    img, gt = load_image_pair(img_path, mask_path)
    
    for cs in color_spaces:
        for k in k_values:
            mask, _ = segment_kmeans(img, k=k, color_space=cs)
            iou = evaluate_quick(mask, gt)
            tuning_results.append({'k': k, 'color_space': cs, 'iou': iou})

df_tuning = pd.DataFrame(tuning_results)
summary = df_tuning.groupby(['color_space', 'k'])['iou'].mean().unstack()
print("\nMean IoU across first 10 images:")
print(summary.round(4))
print(f"\nBest config: {df_tuning.groupby(['color_space', 'k'])['iou'].mean().idxmax()}")

In [ ]:
# Part 4: Batch Processing
# 
# >>> UPDATE KMEANS_CONFIG BELOW BASED ON TUNING RESULTS <<<

KMEANS_CONFIG = {
    'k': 2,                    # number of clusters - update if tuning says otherwise
    'color_space': 'lab',      # color space - update if tuning says otherwise
    'use_spatial': False,
    'spatial_weight': 0.5,
}

print("kmeans configuration:")
for key, value in KMEANS_CONFIG.items():
    print(f"  {key}: {value}")

# Process all images
print(f"\nprocessing {len(df_manifest)} images with kmeans...")

results = []

for idx, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc="kmeans"):
    # Load image
    img_path = PREPROCESSED_DIR / 'images' / row['filename']
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Segment
    try:
        start_time = time.time()
        mask, _ = segment_kmeans(
            img,
            k=KMEANS_CONFIG['k'],
            color_space=KMEANS_CONFIG['color_space'],
            use_spatial=KMEANS_CONFIG['use_spatial'],
            spatial_weight=KMEANS_CONFIG['spatial_weight']
        )
        elapsed = time.time() - start_time
        
        # Save result
        output_path = RESULTS_DIR / row['mask_filename']
        cv2.imwrite(str(output_path), mask)
        
        results.append({
            'filename': row['filename'],
            'mask_filename': row['mask_filename'],
            'subset': row['subset'],
            'status': 'success',
            'time_seconds': round(elapsed, 3),
            'output_path': str(output_path)
        })
        
    except Exception as e:
        print(f"\nerror processing {row['filename']}: {e}")
        results.append({
            'filename': row['filename'],
            'mask_filename': row['mask_filename'],
            'subset': row['subset'],
            'status': 'failed',
            'error': str(e)
        })

# Save results manifest
df_results = pd.DataFrame(results)
df_results.to_csv(RESULTS_DIR / 'kmeans_results.csv', index=False)

print(f"\nprocessing complete:")
print(f"  success: {(df_results['status'] == 'success').sum()}")
print(f"  failed: {(df_results['status'] == 'failed').sum()}")
if 'time_seconds' in df_results.columns:
    print(f"  avg time per image: {df_results['time_seconds'].mean():.3f}s")
print(f"  results saved to: {RESULTS_DIR}")

In [ ]:
# Part 5: Qualitative Validation

n_samples_per_subset = 2
validation_samples = []

for subset in sorted(df_manifest['subset'].unique()):
    subset_data = df_results[df_results['subset'] == subset]
    successful = subset_data[subset_data['status'] == 'success']
    
    if len(successful) > 0:
        samples = successful.sample(min(n_samples_per_subset, len(successful)), random_state=42)
        validation_samples.extend(samples.index.tolist())

print(f"visualizing {len(validation_samples)} validation samples...")

for sample_idx in validation_samples:
    row = df_results.iloc[sample_idx]
    
    img_path = PREPROCESSED_DIR / 'images' / row['filename']
    gt_path = PREPROCESSED_DIR / 'masks' / row['mask_filename']
    pred_path = RESULTS_DIR / row['mask_filename']
    
    img, gt_mask = load_image_pair(img_path, gt_path)
    pred_mask = cv2.imread(str(pred_path), cv2.IMREAD_GRAYSCALE)
    
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    
    axes[0].imshow(img)
    axes[0].set_title('input')
    axes[0].axis('off')
    
    axes[1].imshow(pred_mask, cmap='gray')
    axes[1].set_title('kmeans prediction')
    axes[1].axis('off')
    
    axes[2].imshow(gt_mask, cmap='gray')
    axes[2].set_title('ground truth')
    axes[2].axis('off')
    
    overlay = img.copy()
    overlay[pred_mask > 127] = overlay[pred_mask > 127] * 0.5 + np.array([0, 255, 0]) * 0.5
    axes[3].imshow(overlay.astype(np.uint8))
    axes[3].set_title('overlay (green=pred)')
    axes[3].axis('off')
    
    plt.suptitle(f'Validation - {row["subset"]} - {row["filename"][:50]}', fontsize=11)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'validation_{sample_idx}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Part 6: Save Metadata

metadata = {
    'method': 'kmeans',
    'config': KMEANS_CONFIG,
    'total_images': int(len(df_manifest)),
    'successful': int((df_results['status'] == 'success').sum()),
    'failed': int((df_results['status'] == 'failed').sum()),
    'timestamp': pd.Timestamp.now().isoformat()
}

with open(RESULTS_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("metadata saved")

In [ ]:
# Summary

print("\n" + "="*60)
print("K-MEANS SEGMENTATION COMPLETE")
print("="*60)
print(f"method: kmeans")
print(f"k: {KMEANS_CONFIG['k']}")
print(f"color space: {KMEANS_CONFIG['color_space']}")
print(f"images processed: {len(df_manifest)}")
print(f"success rate: {(df_results['status'] == 'success').sum() / len(df_manifest) * 100:.1f}%")
print(f"results: {RESULTS_DIR}")
print("="*60)
print("\nNext step: run notebook 03 with METHOD_CONFIG name='kmeans'")

## **Output Structure:**
```
results/kmeans/
├── DIS-TE1_*.png              # 20 segmentation masks
├── DIS-TE2_*.png              # 20 segmentation masks
├── DIS-TE3_*.png              # 20 segmentation masks
├── DIS-TE4_*.png              # 20 segmentation masks
├── kmeans_results.csv         # processing manifest
├── metadata.json              # configuration
├── colorspace_comparison.png
├── k_comparison.png
└── validation_*.png
```